# Kaggle Competition: Facial Feature Classification Challenge

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

data_folder = Path("../enge-707-2026-s-2-cat-and-dog-facial-feature-classification-ch")  # Competition data folder (relative to this notebook)

training_images_folder = data_folder / "train" / "train"                    # Training images folder path
testing_images_folder = data_folder / "test" / "test"                       # Testing images folder path

sample_submission_csv_path = data_folder / "sample_submission.csv"          # Sample submission CSV file path
train_labels_csv_path = data_folder / "train_labels.csv"                    # Training labels CSV file path

df_sample_submission = pd.read_csv(sample_submission_csv_path)  # Load sample submission CSV into a DataFrame
df_train_labels = pd.read_csv(train_labels_csv_path)            # Load training labels CSV into a DataFrame

## Image Analysis

Collect per-image information for every image in the train and test folders (file size, dimensions, pixel count, colour mode, brightness/colour statistics, duplicates) and save it to `images_info.csv`.

In [ ]:
import hashlib
from PIL import Image

images_info_csv_path = Path("images_info.csv")  # Output CSV file path


def get_image_info(image_path, split):
    """Return a dictionary of information about a single image."""
    file_bytes = image_path.read_bytes()
    info = {
        "split": split,                                         # train or test
        "image_id": int(image_path.stem),                       # Matches image_id in the CSV files
        "file_name": image_path.name,
        "file_size_bytes": len(file_bytes),
        "file_size_kb": round(len(file_bytes) / 1024, 3),
        "md5_hash": hashlib.md5(file_bytes).hexdigest(),        # Used to find duplicate images
    }
    try:
        with Image.open(image_path) as img:
            info["format"] = img.format
            info["mode"] = img.mode                             # e.g. RGB, L (greyscale), RGBA
            info["width"] = img.width
            info["height"] = img.height
            info["channels"] = len(img.getbands())

            pixels = np.asarray(img.convert("RGB"), dtype=np.float32)
            grey = pixels.mean(axis=2)
            info["mean_brightness"] = round(float(grey.mean()), 3)      # 0 (black) to 255 (white)
            info["std_brightness"] = round(float(grey.std()), 3)        # Contrast
            info["min_pixel"] = int(pixels.min())
            info["max_pixel"] = int(pixels.max())
            info["mean_red"] = round(float(pixels[:, :, 0].mean()), 3)
            info["mean_green"] = round(float(pixels[:, :, 1].mean()), 3)
            info["mean_blue"] = round(float(pixels[:, :, 2].mean()), 3)
            # Greyscale images stored as RGB have identical channels
            info["is_greyscale"] = bool(np.array_equal(pixels[:, :, 0], pixels[:, :, 1]) and np.array_equal(pixels[:, :, 1], pixels[:, :, 2]))
        info["is_corrupt"] = False
    except Exception:
        info["is_corrupt"] = True
    return info


image_records = []
for split, folder in [("train", training_images_folder), ("test", testing_images_folder)]:
    image_paths = sorted(folder.glob("*.jpg"))
    print(f"Analysing {len(image_paths)} {split} images...")
    image_records.extend(get_image_info(path, split) for path in image_paths)

df_images_info = pd.DataFrame(image_records)

# Derived columns
df_images_info["total_pixels"] = df_images_info["width"] * df_images_info["height"]
df_images_info["aspect_ratio"] = (df_images_info["width"] / df_images_info["height"]).round(3)
df_images_info["is_duplicate"] = df_images_info.duplicated("md5_hash", keep=False)    # Exact copies anywhere in train/test

# Add the class label for training images (test images have no label)
df_train_labels["image_id"] = df_train_labels["image_id"].astype(int)
df_images_info = df_images_info.merge(
    df_train_labels.assign(split="train"), on=["split", "image_id"], how="left"
)

column_order = [
    "split", "image_id", "file_name", "class", "format", "mode", "channels",
    "width", "height", "total_pixels", "aspect_ratio", "file_size_bytes", "file_size_kb",
    "mean_brightness", "std_brightness", "min_pixel", "max_pixel",
    "mean_red", "mean_green", "mean_blue", "is_greyscale", "is_corrupt", "is_duplicate", "md5_hash",
]
df_images_info = df_images_info[column_order]
df_images_info.to_csv(images_info_csv_path, index=False)
print(f"Saved {len(df_images_info)} rows to {images_info_csv_path}")

# Quick summary
print("\nImages per split:")
print(df_images_info["split"].value_counts())
print("\nColour modes:")
print(df_images_info.groupby(["split", "mode"]).size())
print(f"\nUnique (width, height) sizes: {df_images_info.groupby(['width', 'height']).ngroups}")
print(f"Greyscale images: {df_images_info['is_greyscale'].sum()}")
print(f"Corrupt images: {df_images_info['is_corrupt'].sum()}")
print(f"Duplicate images: {df_images_info['is_duplicate'].sum()}")
print("\nSize summary per split:")
print(df_images_info.groupby("split")[["width", "height", "total_pixels", "file_size_kb"]].describe().T)
df_images_info.describe()